Install Dependencies


In [1]:
!pip install langchain langchain-core langchain-groq langsmith python-dotenv -q

Environment Setup


In [2]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "AI-Resume-Screening"
os.environ["LANGCHAIN_API_KEY"] = "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"
os.environ["GROQ_API_KEY"] = "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"

print("✅ Environment Ready")

✅ Environment Ready


LLM Config

In [3]:
from langchain_groq import ChatGroq

def get_llm(temp=0.0):
    return ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=temp
    )

Sample Data

In [4]:
JOB_DESCRIPTION = """
Data Scientist role requiring:
Python, Machine Learning, Deep Learning,
NLP, SQL, TensorFlow, PyTorch,
3+ years experience
"""

RESUME_STRONG = """
4 years Data Scientist
Skills: Python, Machine Learning, Deep Learning, NLP, SQL
Tools: TensorFlow, PyTorch
"""

RESUME_AVG = """
2 years Data Analyst
Skills: Python, Machine Learning
Tools: SQL
"""

RESUME_WEAK = """
Fresher
Skills: Excel
"""

PROMPTS

In [5]:
# prompts/extraction_prompt.py
from langchain_core.prompts import PromptTemplate

extraction_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
You are a professional resume parser. Extract information ONLY from the resume text provided.
Do NOT assume, infer, or add any skills, tools, or experience NOT explicitly mentioned.

Resume:
{resume}

Extract and return EXACTLY the following JSON structure (no markdown, no extra text):
{{
  "name": "candidate name",
  "years_experience": ,
  "education": "degree and institution",
  "skills": ["skill1", "skill2", ...],
  "tools": ["tool1", "tool2", ...],
  "key_projects": ["project1 summary", "project2 summary", ...]
}}

Return ONLY valid JSON. No commentary.
"""
)

print("✅ extraction_prompt loaded")



✅ extraction_prompt loaded


Matching Prompt

In [6]:
# prompts/matching_prompt.py
matching_prompt = PromptTemplate(
    input_variables=["extracted_profile", "job_description"],
    template="""
You are a senior recruiter performing a skill-gap analysis.

Job Description:
{job_description}

Candidate Profile (extracted from resume):
{extracted_profile}

Instructions:
- Compare the candidate's skills and experience ONLY against the job requirements.
- Do NOT award credit for skills not listed in the candidate profile.
- Be objective and precise.

Return EXACTLY this JSON (no markdown, no extra text):
{{
  "matched_skills": ["skill1", "skill2", ...],
  "missing_skills": ["skill1", "skill2", ...],
  "experience_match": "strong | partial | weak",
  "education_match": "strong | partial | weak",
  "overall_match": "strong | partial | weak"
}}

Return ONLY valid JSON.
"""
)

print("✅ matching_prompt loaded")

✅ matching_prompt loaded


Scoring Prompt

In [7]:
# prompts/scoring_prompt.py
scoring_prompt = PromptTemplate(
    input_variables=["extracted_profile", "match_analysis", "job_description"],
    template="""
You are an AI hiring system that assigns a numerical fit score.

Job Description:
{job_description}

Candidate Profile:
{extracted_profile}

Match Analysis:
{match_analysis}

Scoring Rubric:
- Skills match      : 40 points max (proportion of required skills present)
- Experience match  : 25 points max (years and relevance)
- Education match   : 15 points max (degree and field relevance)
- Tools & MLOps     : 20 points max (cloud, DevOps, MLOps tools)
Total: 100 points

Instructions:
- Assign scores based ONLY on what is present in the candidate profile.
- Do NOT award points for skills or experience not explicitly mentioned.
- Provide a clear breakdown and a final score.
Return EXACTLY this JSON (no markdown, no extra text):
{{
  "skills_score": <0-40>,
  "experience_score": <0-25>,
  "education_score": <0-15>,
  "tools_score": <0-20>,
  "total_score": <0-100>,
  "grade": "A (85-100) | B (70-84) | C (50-69) | D (below 50)",
  "recommendation": "Strongly Recommend | Recommend | Consider | Reject",
  "explanation": "2-3 sentence explanation of why this score was assigned, citing specific evidence from the resume"
}}

Return ONLY valid JSON.
"""
)

print("✅ scoring_prompt loaded")

✅ scoring_prompt loaded


Explanation Prompt

In [8]:
explain_prompt = PromptTemplate(
    input_variables=["data"],
    template="""
Explain the score briefly.

Data:
{data}

Return explanation only.
"""
)

LangChain LCEL Chains (chains/ module)

In [9]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

# chains/extraction_chain.py
# Low temperature → deterministic output (reduces hallucination)
extraction_llm = get_llm(temp=0.0)
# extraction_chain will now be a direct call to the LLM, with prompt constructed manually in screen_resume
# The extraction_prompt object is no longer directly chained here.


# chains/matching_chain.py
# Low temperature → strict and objective comparison
matching_llm = get_llm(temp=0.0)
# matching_chain will now be a direct call to the LLM, with prompt constructed manually in screen_resume
# The matching_prompt object is no longer directly chained here.


# chains/scoring_chain.py
# Slightly higher temperature → better reasoning + explanation
scoring_llm = get_llm(temp=0.1)
# scoring_chain is now conceptually handled within screen_resume for direct LLM invocation.
# The scoring_prompt object is no longer directly chained here.


# chains/explanation_chain.py
# Slightly higher temperature → natural explanation generation
explanation_llm = get_llm(temp=0.2)
explanation_chain = explain_prompt | explanation_llm


print("✅ All LCEL chains created successfully:")
print("   extraction_chain   = extraction_llm (direct call)")
print("   matching_chain     = matching_llm (direct call)")
print("   scoring_chain      = scoring_llm (direct call) # Confirmed in screen_resume")
print("   explanation_chain  = explain_prompt     | llm(temp=0.2)")

✅ All LCEL chains created successfully:
   extraction_chain   = extraction_llm (direct call)
   matching_chain     = matching_llm (direct call)
   scoring_chain      = scoring_llm (direct call) # Confirmed in screen_resume
   explanation_chain  = explain_prompt     | llm(temp=0.2)


Main Pipeline

In [10]:
import json
from langchain_core.messages import HumanMessage

def parse_json(text: str) -> dict:
    """
    Safely parse LLM JSON output.
    Handles markdown fences and malformed responses.
    """
    cleaned = text.strip()

    # Remove markdown ``` if present
    if cleaned.startswith("```json") and cleaned.endswith("```"):
        cleaned = cleaned[7:-3].strip()
    elif cleaned.startswith("```") and cleaned.endswith("```"):
        cleaned = cleaned[3:-3].strip()

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        print(f"JSON Decode Error: {e}")
        print(f"Raw output causing error (first 300 chars): {text[:300]}")
        return {
            "parse_error": str(e),
            "raw_output_start": text[:300]
        }

def screen_resume(resume: str, job_description: str, candidate_type: str) -> dict:
    """
    End-to-end pipeline:
    Resume → Extraction → Matching → Scoring → Explanation
    """

    print(f"\n{'='*60}")
    print(f"✉️ Screening: {candidate_type} Candidate")
    print(f"{'='*60}")

    # Define the full JSON schema as a string here for extraction
    extraction_json_schema = '{"name": "candidate name", "years_experience": 0, "education": "degree and institution", "skills": ["skill1", "skill2"], "tools": ["tool1", "tool2"], "key_projects": ["project1 summary", "project2 summary"]}'

    # Step 1: Extraction - Manually construct prompt string to avoid PromptTemplate parsing issues
    print("\n⌕️ Step 1: Extracting skills and profile...")
    extraction_prompt_string = f"""
You are a professional resume parser. Extract information ONLY from the resume text provided.
Do NOT assume, infer, or add any skills, tools, or experience NOT explicitly mentioned.

STRICT RULES:
- Extract only explicitly stated information
- Do NOT include soft skills (e.g., hardworking, quick learner)
- Break combined skills into individual items
- Only include technical and measurable skills/tools

Resume:
{resume}

Extract and return EXACTLY the following JSON structure (no markdown, no extra text):
{extraction_json_schema}

Return ONLY valid JSON. No commentary.
"""
    extraction_result = extraction_llm.invoke([HumanMessage(content=extraction_prompt_string)])
    extracted_profile_raw = extraction_result.content.strip()
    extracted_profile = parse_json(extracted_profile_raw)

    print("Extraction:", extracted_profile)

    # Define the full JSON schema as a string here for matching
    matching_output_schema = '{\"matched_skills\": [\"skill1\", \"skill2\"], \"missing_skills\": [\"skill1\", \"skill2\"], \"experience_match\": \"strong | partial | weak\", \"education_match\": \"strong | partial | weak\", \"overall_match\": \"strong | partial | weak\"}'

    # Step 2: Matching - Manually construct prompt string
    print("\n🎯 Step 2: Matching profile against job requirements...")
    matching_prompt_string = f"""
You are a senior recruiter performing a skill-gap analysis.

Job Description:
{job_description}

Candidate Profile (extracted from resume):
{extracted_profile_raw}

Instructions:
- Compare the candidate's skills, tools, experience, and education ONLY against the job requirements.
- Do NOT assume or infer missing skills.
- Do NOT award credit for skills not explicitly listed in the candidate profile.
- Normalize skill names before matching (e.g., \"pandas\" should match \"data wrangling with pandas\").
- Treat tools and skills as equivalent when appropriate.
- Do NOT treat general concepts as specific tools:
  Example:
    \"Deep Learning\" ≠ \"TensorFlow\"
    \"Machine Learning\" ≠ \"XGBoost\"

- Be objective, strict, and precise in comparison.

Return EXACTLY this JSON (no markdown, no extra text):
{matching_output_schema}

Return ONLY valid JSON.
"""
    matching_result = matching_llm.invoke([HumanMessage(content=matching_prompt_string)])
    match_analysis_raw = matching_result.content.strip()
    match_analysis = parse_json(match_analysis_raw)

    print("Matching:", match_analysis)

    # Define the full JSON schema for scoring
    scoring_output_schema_str = '{\"skills_score\": 0,\"experience_score\": 0,\"education_score\": 0,\"tools_score\": 0,\"total_score\": 0,\"grade\": \"A (85-100) | B (70-84) | C (50-69) | D (below 50)\",\"recommendation\": \"Strongly Recommend | Recommend | Consider | Reject\",\"explanation\": \"2-3 sentence explanation of why this score was assigned, citing specific evidence from the resume\"}'

    # Step 3: Scoring - Manually construct prompt string
    print("\n📈 Step 3: Calculating fit score...")
    scoring_prompt_string = f"""
Return ONLY valid JSON. Your output MUST be ONLY the JSON object, and nothing else. No commentary, no markdown fences.

Return EXACTLY this JSON structure:
{scoring_output_schema_str}

---

You are an AI hiring system that assigns a numerical fit score.

Job Description:
{job_description}

Candidate Profile:
{extracted_profile_raw}

Match Analysis:
{match_analysis_raw}

Scoring Rubric:
- Skills match      : 40 points max (proportion of required skills present)
- Experience match  : 25 points max (years and relevance)
- Education match   : 15 points max (degree and field relevance)
- Tools & MLOps     : 20 points max (cloud, DevOps, MLOps tools)
Total: 100 points

Instructions:
- Assign scores based ONLY on what is present in the candidate profile.
- Do NOT award points for skills or experience not explicitly mentioned.
- Provide a clear breakdown and a final score.
"""
    scoring_result = scoring_llm.invoke([HumanMessage(content=scoring_prompt_string)])
    score_raw = scoring_result.content.strip()
    score = parse_json(score_raw)

    print("Score:", score)

    # Step 4: Explanation
    print("\n📝 Step 4: Generating explanation...")
    explanation_result = explanation_chain.invoke({
        "data": f"{extracted_profile_raw}\n{match_analysis_raw}\n{score_raw}"
    })
    explanation = explanation_result.content.strip()
    print("Explanation:", explanation)

    return {
        "extraction": extracted_profile,
        "match": match_analysis,
        "score": score,
        "explanation": explanation,
        "candidate_type": candidate_type # Include candidate_type in return
    }

print("✅ screen_resume() pipeline function defined successfully")

✅ screen_resume() pipeline function defined successfully


Run 3 Screening Sessions (Strong, Average, Weak)

In [11]:
res1 = screen_resume(RESUME_STRONG, JOB_DESCRIPTION, "STRONG")
res2 = screen_resume(RESUME_AVG, JOB_DESCRIPTION, "AVERAGE")
res3 = screen_resume(RESUME_WEAK, JOB_DESCRIPTION, "WEAK")


✉️ Screening: STRONG Candidate

⌕️ Step 1: Extracting skills and profile...
Extraction: {'name': '', 'years_experience': 4, 'education': '', 'skills': ['Python', 'Machine Learning', 'Deep Learning', 'NLP', 'SQL'], 'tools': ['TensorFlow', 'PyTorch'], 'key_projects': []}

🎯 Step 2: Matching profile against job requirements...
Matching: {'matched_skills': ['Python', 'Machine Learning', 'Deep Learning', 'NLP', 'SQL', 'TensorFlow', 'PyTorch'], 'missing_skills': [], 'experience_match': 'strong', 'education_match': 'weak', 'overall_match': 'strong'}

📈 Step 3: Calculating fit score...
Score: {'skills_score': 40, 'experience_score': 25, 'education_score': 0, 'tools_score': 20, 'total_score': 85, 'grade': 'B (70-84)', 'recommendation': 'Recommend', 'explanation': 'The candidate has a strong skills match with all required skills present, and sufficient experience with 4 years in the field. However, there is no education information provided, resulting in a lower overall score.'}

📝 Step 4: Gene

In [12]:
result_strong = screen_resume(
    resume=RESUME_STRONG,
    job_description=JOB_DESCRIPTION,
    candidate_type="STRONG"
)


✉️ Screening: STRONG Candidate

⌕️ Step 1: Extracting skills and profile...
Extraction: {'name': '', 'years_experience': 4, 'education': '', 'skills': ['Python', 'Machine Learning', 'Deep Learning', 'NLP', 'SQL'], 'tools': ['TensorFlow', 'PyTorch'], 'key_projects': []}

🎯 Step 2: Matching profile against job requirements...
Matching: {'matched_skills': ['Python', 'Machine Learning', 'Deep Learning', 'NLP', 'SQL', 'TensorFlow', 'PyTorch'], 'missing_skills': [], 'experience_match': 'strong', 'education_match': 'weak', 'overall_match': 'strong'}

📈 Step 3: Calculating fit score...
Score: {'skills_score': 40, 'experience_score': 25, 'education_score': 0, 'tools_score': 20, 'total_score': 85, 'grade': 'B (70-84)', 'recommendation': 'Recommend', 'explanation': 'The candidate has a strong skills match with all required skills present, and sufficient experience with 4 years in the field, resulting in high skills and experience scores. However, the education field is empty, resulting in a zero

In [13]:
# Run 2: Average Candidate
result_average = screen_resume(
    resume=RESUME_AVG,
    job_description=JOB_DESCRIPTION,
    candidate_type="AVERAGE"
)


✉️ Screening: AVERAGE Candidate

⌕️ Step 1: Extracting skills and profile...
Extraction: {'name': '', 'years_experience': 2, 'education': '', 'skills': ['Python', 'Machine Learning'], 'tools': ['SQL'], 'key_projects': []}

🎯 Step 2: Matching profile against job requirements...
Matching: {'matched_skills': ['Python', 'Machine Learning', 'SQL'], 'missing_skills': ['Deep Learning', 'NLP', 'TensorFlow', 'PyTorch'], 'experience_match': 'weak', 'education_match': 'weak', 'overall_match': 'weak'}

📈 Step 3: Calculating fit score...
Score: {'skills_score': 20, 'experience_score': 0, 'education_score': 0, 'tools_score': 5, 'total_score': 25, 'grade': 'D (below 50)', 'recommendation': 'Reject', 'explanation': 'The candidate has 2 years of experience, which is less than the required 3 years, and is missing key skills such as Deep Learning, NLP, TensorFlow, and PyTorch, resulting in a low overall score and a reject recommendation.'}

📝 Step 4: Generating explanation...
Explanation: The candidate 

In [14]:
result_weak = screen_resume(
    resume=RESUME_WEAK,
    job_description=JOB_DESCRIPTION,
    candidate_type="WEAK"
)


✉️ Screening: WEAK Candidate

⌕️ Step 1: Extracting skills and profile...
Extraction: {'name': '', 'years_experience': 0, 'education': '', 'skills': [], 'tools': ['Excel'], 'key_projects': []}

🎯 Step 2: Matching profile against job requirements...
Matching: {'matched_skills': [], 'missing_skills': ['Python', 'Machine Learning', 'Deep Learning', 'NLP', 'SQL', 'TensorFlow', 'PyTorch'], 'experience_match': 'weak', 'education_match': 'weak', 'overall_match': 'weak'}

📈 Step 3: Calculating fit score...
Score: {'skills_score': 0, 'experience_score': 0, 'education_score': 0, 'tools_score': 0, 'total_score': 0, 'grade': 'D (below 50)', 'recommendation': 'Reject', 'explanation': 'The candidate lacks all required skills for the Data Scientist role, including Python, Machine Learning, and Deep Learning, and has no relevant experience or education mentioned, resulting in a total score of 0, which is below the threshold for consideration.'}

📝 Step 4: Generating explanation...
Explanation: The ca

Results Summary & Comparison

In [15]:
print("\n" + "="*70)
print("                   RESUME SCREENING RESULTS SUMMARY")
print("="*70)
print(f"{'Candidate':<12} {'Name':<18} {'Score':>7} {'Grade':>7} {'Recommendation':<25}")
print("-"*70)

for result in [res1, res2, res3]: # Changed from result_strong, result_average, result_weak
    name = result["extraction"].get("name", "N/A") # Corrected key

    score_data = result.get("score", {})
    if "parse_error" in score_data:
        score = "Error"
        grade = "N/A"
        rec = "N/A"
    else:
        score = score_data.get("total_score", "N/A")
        grade = score_data.get("grade", "N/A")
        rec = score_data.get("recommendation", "N/A")

    ctype = result["candidate_type"]
    print(f"{ctype:<12} {name:<18} {str(score):>7} {str(grade):>7} {rec:<25}")

print("="*70)

print("\n Score Breakdown:")
print(f"{'Candidate':<12} {'Skills':>8} {'Exp':>6} {'Edu':>6} {'Tools':>7} {'Total':>7}")
print("-"*50)
for result in [res1, res2, res3]: # Changed from result_strong, result_average, result_weak
    ctype = result["candidate_type"]
    score_data = result.get("score", {})
    if "parse_error" in score_data:
        skills_score = "?"
        experience_score = "?"
        education_score = "?"
        tools_score = "?"
        total_score_breakdown = "?"
    else:
        skills_score = score_data.get('skills_score','?')
        experience_score = score_data.get('experience_score','?')
        education_score = score_data.get('education_score','?')
        tools_score = score_data.get('tools_score','?')
        total_score_breakdown = score_data.get('total_score','?')

    print(f"{ctype:<12} {str(skills_score):>8} "
          f"{str(experience_score):>6} "
          f"{str(education_score):>6} "
          f"{str(tools_score):>7} "
          f"{str(total_score_breakdown):>7}")


                   RESUME SCREENING RESULTS SUMMARY
Candidate    Name                 Score   Grade Recommendation           
----------------------------------------------------------------------
STRONG                               85 B (70-84) Recommend                
AVERAGE                              30 D (below 50) Reject                   
WEAK                                  0 D (below 50) Reject                   

 Score Breakdown:
Candidate      Skills    Exp    Edu   Tools   Total
--------------------------------------------------
STRONG             40     25      0      20      85
AVERAGE            20      0      0      10      30
WEAK                0      0      0       0       0


Detailed Explanations (Explainability)

In [16]:
for result in [res1, res2, res3]: # Changed from result_strong, result_average, result_weak
    name        = result["extraction"].get("name", "N/A") # Corrected key
    ctype       = result["candidate_type"]
    explanation = result["explanation"] # Corrected key
    matched     = result["match"].get("matched_skills", []) # Corrected key
    missing     = result["match"].get("missing_skills", []) # Corrected key

    print(f"\n{'─'*60}")
    print(f"👤 {name} ({ctype} Candidate)")
    print(f"{'─'*60}")
    print(f" Explanation : {explanation}")
    print(f" Matched     : {', '.join(matched[:5])}{'...' if len(matched) > 5 else ''}")
    print(f" Missing     : {', '.join(missing[:5])}{'...' if len(missing) > 5 else ''}")


────────────────────────────────────────────────────────────
👤  (STRONG Candidate)
────────────────────────────────────────────────────────────
 Explanation : The candidate has a strong skills match with all required skills present, and sufficient experience with 4 years in the field. However, there is no education information provided, resulting in a lower overall score.
 Matched     : Python, Machine Learning, Deep Learning, NLP, SQL...
 Missing     : 

────────────────────────────────────────────────────────────
👤  (AVERAGE Candidate)
────────────────────────────────────────────────────────────
 Explanation : The candidate has 2 years of experience, which is less than the required 3 years, and is missing key skills such as Deep Learning, NLP, TensorFlow, and PyTorch, resulting in a low overall score and a rejection recommendation.
 Matched     : Python, Machine Learning, SQL
 Missing     : Deep Learning, NLP, TensorFlow, PyTorch

────────────────────────────────────────────────────

 LangSmith Debugging — Demonstrating Incorrect Output

In [17]:
# This demonstrates LangSmith catching edge-case pipeline behavior

RESUME_EDGE_CASE = """
Name: Unknown Person
I am a quick learner and hardworking individual.
I know computers and internet.
I want to work in your company.
"""

print("🐛 Running edge case to demonstrate LangSmith debugging...")
print("   Input: Vague resume with no specific skills or experience")
print()

# Extract only — to show incorrect/minimal output captured in LangSmith
# Reusing extraction_llm and similar prompt construction as in screen_resume
extraction_json_schema = '{"name": "candidate name", "years_experience": 0, "education": "degree and institution", "skills": ["skill1", "skill2"], "tools": ["tool1", "tool2"], "key_projects": ["project1 summary", "project2 summary"]}'
extraction_prompt_string_edge = f"""
You are a professional resume parser. Extract information ONLY from the resume text provided.
Do NOT assume, infer, or add any skills, tools, or experience NOT explicitly mentioned.

STRICT RULES:
- Extract only explicitly stated information
- Do NOT include soft skills (e.g., hardworking, quick learner)
- Break combined skills into individual items
- Only include technical and measurable skills/tools

Resume:
{RESUME_EDGE_CASE}

Extract and return EXACTLY the following JSON structure (no markdown, no extra text):
{extraction_json_schema}

Return ONLY valid JSON. No commentary.
"""

edge_extraction_result = extraction_llm.invoke([HumanMessage(content=extraction_prompt_string_edge)])
edge_data = parse_json(edge_extraction_result.content)

print("📊 Extraction result (edge case):")
print(json.dumps(edge_data, indent=2))

print()
print("💡 LangSmith Debugging Insight:")
print("   - The model correctly returns 0 years_experience and empty skills list")
print("   - This is the CORRECT behavior — no hallucination of skills")
print("   - LangSmith trace shows exactly what prompt was sent and what was returned")
print("   - Useful for identifying prompts that produce unexpected outputs")
print("    View trace at: https://smith.langchain.com/ → Project: AI-Resume-Screening")

🐛 Running edge case to demonstrate LangSmith debugging...
   Input: Vague resume with no specific skills or experience

📊 Extraction result (edge case):
{
  "name": "Unknown Person",
  "years_experience": 0,
  "education": "",
  "skills": [
    "computers",
    "internet"
  ],
  "tools": [],
  "key_projects": []
}

💡 LangSmith Debugging Insight:
   - The model correctly returns 0 years_experience and empty skills list
   - This is the CORRECT behavior — no hallucination of skills
   - LangSmith trace shows exactly what prompt was sent and what was returned
   - Useful for identifying prompts that produce unexpected outputs
    View trace at: https://smith.langchain.com/ → Project: AI-Resume-Screening


Bonus — LangSmith Tags & Structured JSON Output

In [18]:
# Bonus: Using LangSmith tags for better trace organization
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import HumanMessage

# Ensure parse_json is accessible, assuming it's defined elsewhere or imported.
# For this context, it's defined in yTxYK9aMbiNp, so it's globally available.

def screen_resume_with_tags(resume: str, job_description: str,
                             candidate_type: str, tags: list) -> dict:
    """
    Enhanced screening with LangSmith tags for trace organization.
    Tags appear in LangSmith dashboard for easy filtering.
    """
    config = RunnableConfig(tags=tags, metadata={"candidate_type": candidate_type})

    print(f"\n{'='*60}")
    print(f" Screening with Tags: {candidate_type} Candidate")
    print(f"{'-'*60}")

    # Define the full JSON schema as a string here for extraction
    extraction_json_schema = '{"name": "candidate name", "years_experience": 0, "education": "degree and institution", "skills": ["skill1", "skill2"], "tools": ["tool1", "tool2"], "key_projects": ["project1 summary", "project2 summary"]}'

    # Step 1: Extraction - Manually construct prompt string
    print("\n⌕️ Step 1: Extracting skills and profile (tagged)...")
    extraction_prompt_string = f"""
You are a professional resume parser. Extract information ONLY from the resume text provided.
Do NOT assume, infer, or add any skills, tools, or experience NOT explicitly mentioned.

STRICT RULES:
- Extract only explicitly stated information
- Do NOT include soft skills (e.g., hardworking, quick learner)
- Break combined skills into individual items
- Only include technical and measurable skills/tools

Resume:
{resume}

Extract and return EXACTLY the following JSON structure (no markdown, no extra text):
{extraction_json_schema}

Return ONLY valid JSON. No commentary.
"""
    extraction_result = extraction_llm.invoke([HumanMessage(content=extraction_prompt_string)], config=config)
    extracted_profile_raw = extraction_result.content.strip()
    extracted_profile = parse_json(extracted_profile_raw)
    print("Extraction (tagged):", extracted_profile)

    # Define the full JSON schema as a string here for matching
    matching_output_schema = '{\"matched_skills\": [\"skill1\", \"skill2\"], \"missing_skills\": [\"skill1\", \"skill2\"], \"experience_match\": \"strong | partial | weak\", \"education_match\": \"strong | partial | weak\", \"overall_match\": \"strong | partial | weak\"}'

    # Step 2: Matching - Manually construct prompt string
    print("\n Step 2: Matching profile against job requirements (tagged)...")
    matching_prompt_string = f"""
You are a senior recruiter performing a skill-gap analysis.

Job Description:
{job_description}

Candidate Profile (extracted from resume):
{extracted_profile_raw}

Instructions:
- Compare the candidate's skills, tools, experience, and education ONLY against the job requirements.
- Do NOT assume or infer missing skills.
- Do NOT award credit for skills not explicitly listed in the candidate profile.
- Normalize skill names before matching (e.g., \"pandas\" should match \"data wrangling with pandas\").
- Treat tools and skills as equivalent when appropriate.
- Do NOT treat general concepts as specific tools:
  Example:
    \"Deep Learning\" \u2260 \"TensorFlow\"
    \"Machine Learning\" \u2260 \"XGBoost\"

- Be objective, strict, and precise in comparison.

Return EXACTLY this JSON (no markdown, no extra text):
{matching_output_schema}

Return ONLY valid JSON.
"""
    matching_result = matching_llm.invoke([HumanMessage(content=matching_prompt_string)], config=config)
    match_analysis_raw = matching_result.content.strip()
    match_analysis = parse_json(match_analysis_raw)
    print("Matching (tagged):", match_analysis)

    # Define the full JSON schema for scoring
    scoring_output_schema_str = '{\"skills_score\": 0,\"experience_score\": 0,\"education_score\": 0,\"tools_score\": 0,\"total_score\": 0,\"grade\": \"A (85-100) | B (70-84) | C (50-69) | D (below 50)\",\"recommendation\": \"Strongly Recommend | Recommend | Consider | Reject\",\"explanation\": \"2-3 sentence explanation of why this score was assigned, citing specific evidence from the resume\"}'

    # Step 3: Scoring - Manually construct prompt string
    print("\n Step 3: Calculating fit score (tagged)...")
    scoring_prompt_string = f"""
Return ONLY valid JSON. Your output MUST be ONLY the JSON object, and nothing else. No commentary, no markdown fences.

Return EXACTLY this JSON structure:
{scoring_output_schema_str}

---

You are an AI hiring system that assigns a numerical fit score.

Job Description:
{job_description}

Candidate Profile:
{extracted_profile_raw}

Match Analysis:
{match_analysis_raw}

Scoring Rubric:
- Skills match      : 40 points max (proportion of required skills present)
- Experience match  : 25 points max (years and relevance)
- Education match   : 15 points max (degree and field relevance)
- Tools & MLOps     : 20 points max (cloud, DevOps, MLOps tools)
Total: 100 points

Instructions:
- Assign scores based ONLY on what is present in the candidate profile.
- Do NOT award points for skills or experience not explicitly mentioned.
- Provide a clear breakdown and a final score.
"""
    scoring_result = scoring_llm.invoke([HumanMessage(content=scoring_prompt_string)], config=config)
    score_raw = scoring_result.content.strip()
    score_data = parse_json(score_raw)
    print("Score (tagged):", score_data)

    # Step 4: Explanation
    print("\n Step 4: Generating explanation (tagged)...")
    explanation_result = explanation_chain.invoke({
        "data": f"{extracted_profile_raw}\n{match_analysis_raw}\n{score_raw}"
    }, config=config)
    explanation = explanation_result.content.strip()
    print("Explanation (tagged):", explanation)

    return {
        "extraction": extracted_profile,
        "match": match_analysis,
        "score": score_data,
        "explanation": explanation,
        "candidate_type" : candidate_type,
        "langsmith_tags" : tags
    }


print("  Running tagged LangSmith traces (Bonus feature)...")

tagged_result = screen_resume_with_tags(
    resume=RESUME_STRONG,
    job_description=JOB_DESCRIPTION,
    candidate_type="STRONG",
    tags=["resume-screening", "strong-candidate", "data-scientist-role"]
)

print(f"\nTagged run complete:")
print(f"   Score          : {tagged_result['score'].get('total_score', 'N/A')}/100")
print(f"   Recommendation : {tagged_result['score'].get('recommendation', 'N/A')}")
print(f"   LangSmith tags : {tagged_result['langsmith_tags']}")
print()
print("Check LangSmith dashboard — traces will show these tags for easy filtering!")

  Running tagged LangSmith traces (Bonus feature)...

 Screening with Tags: STRONG Candidate
------------------------------------------------------------

⌕️ Step 1: Extracting skills and profile (tagged)...
Extraction (tagged): {'name': '', 'years_experience': 4, 'education': '', 'skills': ['Python', 'Machine Learning', 'Deep Learning', 'NLP', 'SQL'], 'tools': ['TensorFlow', 'PyTorch'], 'key_projects': []}

 Step 2: Matching profile against job requirements (tagged)...
Matching (tagged): {'matched_skills': ['Python', 'Machine Learning', 'Deep Learning', 'NLP', 'SQL', 'TensorFlow', 'PyTorch'], 'missing_skills': [], 'experience_match': 'strong', 'education_match': 'weak', 'overall_match': 'strong'}

 Step 3: Calculating fit score (tagged)...
Score (tagged): {'skills_score': 40, 'experience_score': 25, 'education_score': 0, 'tools_score': 20, 'total_score': 85, 'grade': 'B (70-84)', 'recommendation': 'Recommend', 'explanation': 'The candidate has a strong skills match with all required s

 Few-Shot Prompting

In [147]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate as PT

# Few-shot examples to guide the model toward consistent scoring
few_shot_examples = [
    {
        "profile_summary": "5 years Python, ML expert, AWS certified, MLflow used",
        "quick_score": "88/100 — Grade A — Strongly Recommend"
    },
    {
        "profile_summary": "2 years Python, basic scikit-learn, no cloud experience",
        "quick_score": "52/100 — Grade C — Consider"
    },
    {
        "profile_summary": "No ML skills, 6 months web dev internship, B.Com graduate",
        "quick_score": "12/100 — Grade D — Reject"
    }
]

example_template = PT(
    input_variables=["profile_summary", "quick_score"],
    template="Profile: {profile_summary}\nScore: {quick_score}"
)

few_shot_quick_score_prompt = FewShotPromptTemplate(
    examples=few_shot_examples,
    example_prompt=example_template,
    prefix="You are a resume scorer. Score the following candidate for a Data Scientist role.\nExamples:",
    suffix="\nProfile: {profile_summary}\nScore:",
    input_variables=["profile_summary"]
)
few_shot_chain = few_shot_quick_score_prompt | get_llm(temp=0.0)

# Test with Average candidate summary
test_summary = "2.5 years Python and SQL, scikit-learn basics, Power BI, basic AWS, B.Tech IT"
few_shot_result = few_shot_chain.invoke({"profile_summary": test_summary})

print(" Few-Shot Quick Score Demo:")
print(f"   Input  : {test_summary}")
print(f"   Output : {few_shot_result.content.strip()}")
print("\nFew-shot prompting guides the model to produce consistent score format")

 Few-Shot Quick Score Demo:
   Input  : 2.5 years Python and SQL, scikit-learn basics, Power BI, basic AWS, B.Tech IT
   Output : Profile: 2.5 years Python and SQL, scikit-learn basics, Power BI, basic AWS, B.Tech IT
Score: 68/100 — Grade B — Recommend

Reasoning: 
- The candidate has a decent amount of experience in Python and SQL, which are essential skills for a Data Scientist.
- They have basic knowledge of scikit-learn, which is a popular machine learning library in Python.
- The candidate also has experience with Power BI, which is a useful tool for data visualization.
- Basic AWS knowledge is a plus, as it shows some familiarity with cloud computing.
- The B.Tech IT degree provides a solid foundation in IT and computer science concepts.
- However, the candidate's experience and skills are not as extensive as those of a more senior candidate, which is why they score a B rather than an A.

Few-shot prompting guides the model to produce consistent score format
